# Sentiment analysis of kindle reviews using Word2Vec and RandomForest

##### Workflow
1. Preprocessing and clearning
2. Train test split
3. Word embeddings using Word2Vec
4. Training the model using RandomForest
5. testing the model

### Preprocessing and cleaning

In [1]:
import pandas as pd
df = pd.read_csv('all_kindle_review.csv')
df.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [2]:
df['reviewText'] = df['reviewText'] + df['summary']
df = df[['rating','reviewText']]

In [3]:
df['rating'] = df['rating'].apply(lambda x:0 if x <= 3 else 1) #Positive review is 1, negative is 0

In [4]:
df.head()

,rating,reviewText
0,0,"Jace Rankin may be short, but he's nothing to ..."
1,1,Great short read. I didn't want to put it dow...
2,0,I'll start by saying this is the first of four...
3,0,Aggie is Angela Lansbury who carries pocketboo...
4,1,I did not expect this type of book to be in li...


In [5]:
df.dropna(subset=['reviewText'], inplace=True)

In [6]:
from nltk import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

In [7]:
import re
import nltk
from nltk.stem import WordNetLemmatizer
nltk.download('stopwords')
from nltk.corpus import stopwords
lemmatizer = WordNetLemmatizer()
corpus = []

for i in range(0, len(df)):
    X = re.sub('[^a-zA-Z]', ' ', df['reviewText'].iloc[i])
    X = X.lower()
    X = X.split()
    X = [lemmatizer.lemmatize(word) for word in X if not word in set(stopwords.words('english'))]
    X = ' '.join(X)
    corpus.append(X)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\goggl\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### Train Test Split

In [8]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(
    corpus,
    df['rating'],
    random_state=67,
    test_size=0.25
)

### Word embeddings

In [9]:
from gensim.models import Word2Vec
w2v = Word2Vec(
    sentences=x_train,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4
)

In [10]:
import numpy as np
sentenc_vectors = []

for sent in x_train:
    vectors = []
    for word in sent:
        if word in w2v.wv:
            vectors.append(w2v.wv[word])
    if len(vectors) != 0:
        sentenc_vectors.append(np.mean(vectors, axis=0))
    else:
        sentenc_vectors.append(np.zeros(100))

In [11]:
len(sentenc_vectors)

8998

In [12]:
len(y_train)

8998

### Training the model

In [13]:
from sklearn.ensemble import RandomForestClassifier
classifier = RandomForestClassifier()

In [14]:
classifier.fit(sentenc_vectors, y_train)

RandomForestClassifier()

### Testing the model

In [15]:
import numpy as np
x_test_vec = []

for sent in x_test:
    vectors = []
    for word in sent:
        if word in w2v.wv:
            vectors.append(w2v.wv[word])
    if len(vectors) != 0:
        x_test_vec.append(np.mean(vectors, axis=0))
    else:
        x_test_vec.append(np.zeros(100))

In [16]:
y_pred = classifier.predict(x_test_vec)

In [17]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [18]:
accuracy_score(y_pred=y_pred, y_true=y_test)

0.6086666666666667

In [19]:
print(confusion_matrix(y_pred, y_test))

[[955 674]
 [500 871]]


In [20]:
print(classification_report(y_pred, y_test))

              precision    recall  f1-score   support

           0       0.66      0.59      0.62      1629
           1       0.56      0.64      0.60      1371

    accuracy                           0.61      3000
   macro avg       0.61      0.61      0.61      3000
weighted avg       0.61      0.61      0.61      3000



### Linear Regression

In [21]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()

In [22]:
lr = lr.fit(sentenc_vectors, y_train)

In [23]:
y_pred1 = lr.predict(x_test_vec)

In [24]:
accuracy_score(y_pred=y_pred1, y_true=y_test)

0.6076666666666667

In [25]:
print('Confusion marix is:\n',confusion_matrix(y_pred1, y_test))
print('Classification report is:\n',  classification_report(y_pred1, y_test))

Confusion marix is:
 [[965 687]
 [490 858]]
Classification report is:
               precision    recall  f1-score   support

           0       0.66      0.58      0.62      1652
           1       0.56      0.64      0.59      1348

    accuracy                           0.61      3000
   macro avg       0.61      0.61      0.61      3000
weighted avg       0.61      0.61      0.61      3000

